# Integrated Intelligent Systems: Tutorial 1 - Foundations (3 Pts)

This notebook introduces the fundamental components, challenges, and definitions, central to designing and implementing Cognition-Enabled Robot Agents.

## Part 1: The Unified Theoretical Exercise (E1–E6) (1.5 Pt)

**Title:** From Natural Language Request to action execution: The Eva Coffee Scenario

![Alternative Text](coffee-task-photo.png)

### Scenario Background
Eva requests: "Bring me my coffee." Her usual blue mug is in the dishwasher; a white mug is available on the counter. The fill level can vary, and Eva may shift position during execution.

---

### Multi-Question Student Task Sheet (0.25 Pt x 6)
Students work through this unified exercise in the VRB or on paper, structured across the six developmental steps of Lecture 1:

1. **E1 · Opening Judgment (Data vs. Competence):**
   - If an end-to-end vision-language-action model or a demonstration video shows a robot successfully serving coffee 100 times in a row, does that prove the robot "can do the task" across any home environment? State your reasoning and rate your confidence from 1 to 5.
     

2. **E2 · Changed Case Prediction (Physical Bounds):**
   - A half-full white mug was carried successfully along the route. Suppose the mug is now nearly full. Predict the physical consequence during a turn, and identify the untested assumption in the original run.
     

3. **E3 · The Commitment Decision (Warrant Gate):**
   - The blue mug is unavailable and white-mug acceptance is unconfirmed. Choose whether the robot should ask, observe, or proceed. State the single item of evidence that justifies your choice.
     

4. **E4 · The Five Questions of Understanding (T1–T5):**
   - Formulate one distinct question for each dimension—Explain ($\text{T1}$), Predict ($\text{T2}$), Design ($\text{T3}$), Bound ($\text{T4}$), and Check ($\text{T5}$)—concerning the coffee delivery episode. At least one question must address a changed condition that a video alone cannot answer.
     

5. **E5 · System Decomposition (Mapping to Architecture):**
   - List the core functional modules required to handle Eva's request (interpreting, perceiving, proposing, predicting, warranting, acting, and recording) and map them onto the Integration Diagram.
     

6. **E6 · Transfer & Exit Ticket (Justified Adaptation):**
   - Suppose Eva stands up while the robot is carrying the coffee. Revise the action, predict the potential failure mechanism, and state the exact evidence criterion required before the robot commits to the revised path.

## Part 2: Interactive Virtual Lab & Python Tutorial (1.5 Pt)

**Title:** Implementing the Embodied Cognitive Architecture

### Scenario Background
Eva requests: "Bring me my coffee."

---

In [ ]:
#!/usr/bin/env python3
"""Embodied Cognitive Architecture
"""

from dataclasses import dataclass, field
from typing import Dict, Any, Tuple, Optional, List
import random
import time

@dataclass(frozen=True)
class GoalSpecification:
    """Formalized, concrete goal specification derived via semantic interpretation."""
    target_object: str
    destination: str
    target_volume_ml: float
    max_allowable_force_n: float
    max_slosh_wave_mm: float
    max_nav_velocity_mps: float
    time_frame_seconds: float

@dataclass
class ParameterPacket:
    """System 1 (The Proposer): Flexible sampler proposing execution parameters."""
    grasp_pose: Tuple[float, float, float]
    max_velocity_mps: float
    tilt_rate_radps: float
    source_generator: str = "stochastic_sampler"

class Proposer:
    """ To be completed """
    def __init__(self):
        pass

@dataclass
class Observation:
    """Raw, noisy sensor measurements at time t (distinct from internal belief)."""
    timestamp: float
    measured_temperature_c: float
    measured_friction: float
    eva_position: str
    room_busyness: str

@dataclass
class BeliefState:
    """The robot's current internal belief state (updated via perception)."""
    pouch_present: bool = True
    pouch_material: str = "flexible_plastic"
    fluid_viscosity_pas: float = 1.2
    vial_ready: bool = True
    camera_active: bool = True
    fallback_path_exists: bool = True
    audit_logger_enabled: bool = True
    # Real-time environmental parameters maintained internally
    ambient_temperature_c: float = 21.5
    surface_friction_coefficient: float = 0.45
    eva_position: str = "living_room"
    room_busyness: str = "medium"  # 'low', 'medium', 'high'
    preferred_cup: str = "blue_ceramic_mug"


class InterpretationSystem:
    """Uses a semantic knowledge base to map vague instructions and context 
    into precise physical goal parameters (GoalSpecification).
    """
    def __init__(self, knowledge_base: Dict[str, Any]):
        self.kb = knowledge_base

    def interpret(self, raw_utterance: str, belief: BeliefState) -> GoalSpecification:
        print(f"\n[INTERPRETATION] Analyzing instruction: '{raw_utterance}' in context of Eva (Position: {belief.eva_position}, Room Busyness: {belief.room_busyness})")
        
        cup_type = belief.preferred_cup
        cup_properties = self.kb["cups"].get(cup_type, {"max_force": 5.0, "max_wave": 2.0})
        
        busyness_factor = self.kb["room_busyness_limits"].get(belief.room_busyness, 0.5)
        max_nav_vel = 1.2 * busyness_factor
        
        is_urgent = "schnell" in raw_utterance.lower() or "fast" in raw_utterance.lower()
        time_frame = 30.0 if is_urgent else 60.0
        target_volume = 15.0 if not is_urgent else 12.0
        
        goal = GoalSpecification(
            target_object=cup_type,
            destination=belief.eva_position,
            target_volume_ml=target_volume,
            max_allowable_force_n=cup_properties["max_force"],
            max_slosh_wave_mm=cup_properties["max_wave"],
            max_nav_velocity_mps=max_nav_vel,
            time_frame_seconds=time_frame
        )
        
        print(f" -> Derived Goal: Object={goal.target_object}, MaxForce={goal.max_allowable_force_n}N, MaxNavVel={goal.max_nav_velocity_mps}m/s, TimeFrame={goal.time_frame_seconds}s")
        return goal


class SensingSystem:
    """Simulates real-time noisy hardware sensors emitting raw observations."""
    def __init__(self):
        pass

    def sense(self, true_ambient_temp: float, true_friction: float, eva_pos: str, busyness: str) -> Observation:
        noisy_temp = true_ambient_temp + random.uniform(-0.5, 0.5)
        noisy_friction = true_friction + random.uniform(-0.05, 0.05)
        
        return Observation(
            timestamp=time.time(),
            measured_temperature_c=round(noisy_temp, 2),
            measured_friction=round(noisy_friction, 2),
            eva_position=eva_pos,
            room_busyness=busyness
        )


class PerceptionSystem:
    """Estimator module that updates the internal BeliefState using raw Observations."""
    def __init__(self, belief_state: BeliefState):
        self.belief_state = belief_state

    def update(self, observation: Observation) -> None:
        self.belief_state.ambient_temperature_c = observation.measured_temperature_c
        self.belief_state.surface_friction_coefficient = observation.measured_friction
        self.belief_state.eva_position = observation.eva_position
        self.belief_state.room_busyness = observation.room_busyness


class SemanticDigitalTwin:
    """System 2 (Semantic Digital Twin): Imperfect models with sim-to-real epistemic gaps."""
    def __init__(self, belief_state: BeliefState):
        self.belief = belief_state

    def predict_properties(self, params: ParameterPacket, goal: GoalSpecification) -> Dict[str, float]:
        stiffness_coefficient = 12.5 * (1.0 + self.belief.surface_friction_coefficient * 0.1)
        predicted_force = stiffness_coefficient * (params.max_velocity_mps ** 1.5)
        
        effective_viscosity = self.belief.fluid_viscosity_pas * (1.0 - (self.belief.ambient_temperature_c - 20.0) * 0.01)
        radius = 0.05
        wave_height_m = (params.tilt_rate_radps * radius) / effective_viscosity
        predicted_wave = wave_height_m * 1000.0

        # Injecting minor epistemic simulation error (Sim-to-Real gap)
        epistemic_force_error = random.uniform(-0.2, 0.3)
        epistemic_wave_error = random.uniform(-0.1, 0.2)

        return {
            "predicted_force_n": round(max(0.0, predicted_force + epistemic_force_error), 2),
            "predicted_wave_mm": round(max(0.0, predicted_wave + epistemic_wave_error), 2),
            "predicted_nav_velocity": min(params.max_velocity_mps, goal.max_nav_velocity_mps)
        }


class WarrantGate:
    """Validates predicted properties against the interpreted GoalSpecification."""
    def __init__(self, sdt: SemanticDigitalTwin):
        self.sdt = sdt

    def validate(self, params: ParameterPacket, goal: GoalSpecification, belief: BeliefState) -> Dict[str, Any]:
        predictions = self.sdt.predict_properties(params, goal)
        
        checks = {
            "grounded": belief.vial_ready and belief.pouch_present,
            "feasible": params.max_velocity_mps > 0.0 and params.tilt_rate_radps > 0.0,
            "safe": (predictions["predicted_force_n"] <= goal.max_allowable_force_n) and 
                    (predictions["predicted_wave_mm"] <= goal.max_slosh_wave_mm) and
                    (predictions["predicted_nav_velocity"] <= goal.max_nav_velocity_mps),
            "monitorable": belief.camera_active,
            "repairable": belief.fallback_path_exists,
            "recordable": belief.audit_logger_enabled
        }
        
        all_passed = all(checks.values())
        failed_criterion = next((name for name, passed in checks.items() if not passed), None)
        
        return {
            "passed": all_passed,
            "failed_criterion": failed_criterion,
            "predictions": predictions,
            "warrant_checks": checks
        }


class ExecutionEngine:
    """Manages physical execution and constructs the epistemic audit trail."""
    def __init__(self):
        self.audit_trail: List[Dict[str, Any]] = []

    def execute_action(self, params: ParameterPacket, goal: GoalSpecification, warrant_result: Dict[str, Any]) -> Dict[str, Any]:
        print(f"\n[ACTUATOR] Executing motion in physical world: v_max={params.max_velocity_mps} m/s")
        time.sleep(0.1)
        
        # Real-world divergence check
        predictions = warrant_result["predictions"]
        actual_force = predictions["predicted_force_n"] + random.uniform(-0.3, 0.6)
        actual_wave = predictions["predicted_wave_mm"] + random.uniform(-0.2, 0.5)
        
        spill_occurred = (actual_force > goal.max_allowable_force_n) or (actual_wave > goal.max_slosh_wave_mm)
        status = "FAILURE" if spill_occurred else "SUCCESS"
        
        execution_outcome = {
            "actual_force_n": round(actual_force, 2),
            "actual_wave_mm": round(actual_wave, 2),
            "actual_volume_transferred_ml": goal.target_volume_ml if not spill_occurred else goal.target_volume_ml - 3.0,
            "spill_occurred": spill_occurred,
            "status": status
        }
        return execution_outcome

    def record_trace(self, asked_utterance: str, goal: GoalSpecification, belief: BeliefState, 
                     proposal: ParameterPacket, warrant_result: Dict[str, Any], 
                     observation: Observation, committed: bool, execution_outcome: Optional[Dict[str, Any]] = None) -> None:
        record = {
            "timestamp": time.time(),
            "asked": {
                "raw_utterance": asked_utterance,
                "interpreter_output": {
                    "target": goal.target_object,
                    "max_force_limit_n": goal.max_allowable_force_n,
                    "max_nav_velocity": goal.max_nav_velocity_mps
                }
            },
            "belief": {
                "eva_position": belief.eva_position,
                "room_busyness": belief.room_busyness,
                "ambient_temp_c": belief.ambient_temperature_c,
                "surface_friction": belief.surface_friction_coefficient
            },
            "observed": {
                "timestamp": observation.timestamp,
                "measured_temp_c": observation.measured_temperature_c,
                "measured_friction": observation.measured_friction,
                "eva_position": observation.eva_position,
                "room_busyness": observation.room_busyness
            },
            "committed": {
                "action_approved": committed,
                "proposed_parameters": {
                    "max_velocity_mps": proposal.max_velocity_mps,
                    "tilt_rate_radps": proposal.tilt_rate_radps
                },
                "warrant_verdict": warrant_result
            },
            "execution_outcome": execution_outcome
        }
        self.audit_trail.append(record)


# --- Execution Demonstration ---
if __name__ == "__main__":
    knowledge_base = {
        "cups": {
            "blue_ceramic_mug": {"max_force": 4.5, "max_wave": 1.5},
            "plastic_cup": {"max_force": 7.0, "max_wave": 3.0}
        },
        "room_busyness_limits": {
            "low": 1.0,
            "medium": 0.6,
            "high": 0.3
        }
    }

    # 1. Initialize systems
    belief_state = BeliefState(room_busyness="medium", preferred_cup="blue_ceramic_mug")
    sensing = SensingSystem()
    perception = PerceptionSystem(belief_state)
    interpreter = InterpretationSystem(knowledge_base)
    sdt = SemanticDigitalTwin(belief_state)
    warrant_gate = WarrantGate(sdt)
    executive = ExecutionEngine()
    
    print("=== STEP 1: Sensing & Perception Update ===")
    raw_observation = sensing.sense(
        true_ambient_temp=21.5, 
        true_friction=0.45, 
        eva_pos="living_room", 
        busyness="medium"
    )
    perception.update(raw_observation)
    print(f"Raw Sensor Observation : {raw_observation}")
    print(f"Updated Internal Belief : Temp={belief_state.ambient_temperature_c}°C, Friction={belief_state.surface_friction_coefficient}")
    
    print("\n=== STEP 2: Instruction Interpretation ===")
    raw_command = "Bring me my coffee fast!"
    goal = interpreter.interpret(raw_command, belief_state)
    
    print("\n=== STEP 3: System 1 Proposer Generates Parameters ===")
    proposal_v1 = ParameterPacket(
        grasp_pose=(0.45, 0.12, 0.30),
        max_velocity_mps=0.8, 
        tilt_rate_radps=0.2
    )
    print(f"Proposer suggests: v_max = {proposal_v1.max_velocity_mps} m/s")
    
    print("\n=== STEP 4: System 2 SDT & Warrant Gate Validation (Attempt 1) ===")
    warrant_v1 = warrant_gate.validate(proposal_v1, goal, belief_state)
    print(f"SDT Predictions: Nav-Vel = {warrant_v1['predictions']['predicted_nav_velocity']} m/s")
    print(f"Warrant Checks  : {warrant_v1['warrant_checks']}")
    print(f"Gate Verdict    : REJECTED (Failed on: {warrant_v1['failed_criterion']})")
    
    executive.record_trace(raw_command, goal, belief_state, proposal_v1, warrant_v1, raw_observation, committed=False)
    
    print("\n=== STEP 5: Proposer Resamples (Attempt 2) ===")
    proposal_v2 = ParameterPacket(
        grasp_pose=(0.45, 0.12, 0.30),
        max_velocity_mps=0.45, 
        tilt_rate_radps=0.08
    )
    
    warrant_v2 = warrant_gate.validate(proposal_v2, goal, belief_state)
    print(f"SDT Predictions: Force = {warrant_v2['predictions']['predicted_force_n']} N, Wave = {warrant_v2['predictions']['predicted_wave_mm']} mm")
    print(f"Warrant Checks  : {warrant_v2['warrant_checks']}")
    print(f"Gate Verdict    : APPROVED")
    
    print("\n=== STEP 6: Physical Execution & Epistemic Audit Trail ===")
    if warrant_v2["passed"]:
        final_outcome = executive.execute_action(proposal_v2, goal, warrant_v2)
        executive.record_trace(raw_command, goal, belief_state, proposal_v2, warrant_v2, raw_observation, committed=True, execution_outcome=final_outcome)
        print(f"Execution Outcome: {final_outcome}")

    print(f"\n--- COMPLETE EPISODIC AUDIT TRAIL ({len(executive.audit_trail)} Entries) ---")
    for idx, trace in enumerate(executive.audit_trail, 1):
        status = trace['committed']['action_approved']
        outcome = trace.get('execution_outcome')
        outcome_str = f" | Outcome: {outcome['status']} (Spill: {outcome['spill_occurred']})" if outcome else ""
        print(f"Entry #{idx} | Approved: {status}{outcome_str} | Command: '{trace['asked']['raw_utterance']}'")

## Questions (0.25 Pt x 4 + 0.5 Pt)

### Task 1: Architectural Mapping & Code Decomposition
* **Question 1.1:** Map the classes and methods implemented in your Python script directly onto the functional blocks of the lecture integration diagram. Explain how data flows between these components during a single cognitive decision cycle.
* **Question 1.2:** Complete the proposer class based on a probabilistic model of your choice (e.g., uniform, gaussian, mixture). Justify your choice.
  

### Task 2: Non-Linear Contact Mechanics and Material Compliance
In the base script, the contact force on the flexible pouch is modeled using a simplified power-law equation. 
* **Question 2.1:** Modify the `SemanticDigitalTwin.predict_properties` method to account for a dual-stage material compliance curve. If the contact depth exceeds $0.02 \, \text{m}$, the stiffness coefficient should increase exponentially by a factor of $1.8$ due to bottoming out against a rigid internal fixture.
* **Question 2.2:** Write a test case where the Proposer generates an aggressive approach velocity ($0.35 \, \text{m/s}$). Calculate the predicted force and verify that the Warrant Gate successfully intercepts and rejects the parameter packet before physical execution.

### Task 3: Advanced Fluid Sloshing and Viscosity Temperature Coupling
Fluids in laboratory automation often exhibit temperature-dependent viscosity changes.
* **Question 3.1:** Enhance the fluid dynamics model so that ambient temperature fluctuations directly alter the effective fluid viscosity using an Arrhenius-type approximation or a linear thermal degradation coefficient ($1\% \, \text{drop per} \, ^\circ\text{C}$ above $20^\circ\text{C}$).
* **Question 3.2:** Simulate an environment where the ambient temperature rises to $28^\circ\text{C}$. How does this change the predicted wave height for a tilt rate of $0.15 \, \text{rad/s}$? Does the Warrant Gate trigger a safety violation?

### Task 4: Expanding the Warrant Gate Checklist
The current Warrant Gate evaluates six core criteria (*grounded, feasible, safe, monitorable, repairable, recordable*). 
* **Question 4.1:** Add a seventh criterion called **`energetically_viable`** to the Warrant Gate dictionary checks. It must evaluate whether the estimated battery drain for the requested trajectory stays below $15\%$ of total capacity.
* **Question 4.2:** Introduce a low-battery state into the `BeliefState` and demonstrate how a failed `energetically_viable` check redirects the system workflow to request a recharge action rather than proceeding with the coffee delivery.

### Task 5: Post-Hoc Epistemic Audit Analysis
A core pillar of trustworthy robotics is the ability to audit *why* a decision was made or rejected.
* **Question 5.1:** Write a post-processing function (`audit_analyzer(audit_trail)`) that parses the recorded episodic log. It should compute and print:
  1. The total number of rejected proposals versus approved executions.
  2. The most frequent warrant criterion responsible for rejection.
  3. The average variance between predicted forces and observed physical outcomes.
* **Question 5.2:** Discuss in your lab report: *Why does maintaining separate records for `asked`, `belief`, `committed`, and `observed` solve the epistemological gap that pure data-driven imitation learning cannot address?*

## Part 3: Robot Anatomy (0.5 Pt)

**Title:** Links Vs Joints, Actuators Vs Sensors, Effectors Vs Bases Vs Torsos Vs Heads, Wheels Vs Legs Vs Grippers, Proprioception Vs Exteroception Vs Interoception

![Alternative Text](robot-body.jpg)

### Questions

Identify the key functional parts of a robot based on the above picture and briefly indicate how they support the execution of function in the above architecture.

---

# Submission guidelines

- Deadline: 03.11.2026, 4 PM
- Submission as Git repo commits pushed within deadlines (Create a repo with the name **iis-assignment-win26** for the whole semester and invite me as collaborator)
- Put all the answers in the notebook. Any other media related to your answer should be inserted in the notebook as well. The notebook and all related media should then be put into a folder called **Assignment1**. 